## 0. 환경 설정

In [22]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import torch
from datasets import load_dataset
from collections import Counter
import re
import pickle

## 1. 데이터 로드 및 토크나이저 정의

In [24]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

## 2. 데이터 다운로드 및 vocab 생성


In [25]:
print("load data")

dataset = load_dataset("imdb")
counter = Counter()
for eg in dataset["train"]:
    counter.update(tokenizer(eg["text"]))

vocab_size = 10000
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in counter.most_common(vocab_size - 2):
    vocab[word] = len(vocab)


load data


In [26]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, vocab, max_length = 256):
        self.data = list()
        self.labels = list()
        for eg in hf_dataset:
            tokens = tokenizer(eg["text"])
            encoded = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
            if len(encoded) < max_length:
                encoded += [vocab["<PAD>"]] * (max_length - len(encoded))
            else:
                encoded = encoded[:max_length]
            self.data.append(encoded)
            self.labels.append(eg["label"])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]), torch.tensor(self.labels[idx], dtype = torch.float32)

In [27]:
max_seq_length = 256
train_dataset = IMDBDataset(dataset["train"], vocab, max_length = max_seq_length)
test_dataset = IMDBDataset(dataset["test"], vocab, max_length = max_seq_length)

save_data = {
    "train_dataset": train_dataset,
    "test_dataset": test_dataset,
    "vocab": vocab
}

torch.save(save_data, "/content/drive/MyDrive/DILAB/YB/attention-paper-experiments/Exp2_IMDBdata_hitmap/preprocessed_data.pt")
print("전처리 완료 및 저장")

전처리 완료 및 저장
